### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="seoul_bike_sharing_demand",
    dataset_year="2020",
    domain_str="business & marketing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5F62R",
    download_description="""
    mkdir -p local-data-warehouse/seoul_bike_sharing_demand 
    wget -P local-data-warehouse/seoul_bike_sharing_demand/ https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip
    unzip local-data-warehouse/seoul_bike_sharing_demand/seoul+bike+sharing+demand.zip -d local-data-warehouse/seoul_bike_sharing_demand/ 
    rm local-data-warehouse/seoul_bike_sharing_demand/seoul+bike+sharing+demand.zip
""",
    # References
    academic_reference_bibtex="""@article{sathishkumar2020using,
  title={Using data mining techniques for bike sharing demand prediction in metropolitan city},
  author={Sathishkumar, Veerappampalayam E and Park, Jangwoo and Cho, Yongyun},
  journal={Computer Communications},
  volume={153},
  pages={353--366},
  year={2020},
  publisher={Elsevier}
}

""",
    academic_reference_bibtex_key="sathishkumar2020using",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""

""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="regression",
    problem_type="",
    objective_metric_name="",
    stratify_on="",
)

ValidationError: 1 validation error for PredictiveMLTaskMetadata
problem_type
  Input should be 'binary_classification', 'multiclass_classification' or 'regression' [type=literal_error, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "SeoulBikeData.csv", encoding="unicode_escape")
print("Loaded data shape:", df.shape)

Loaded data shape: (8760, 14)


In [11]:
from data_foundry.temporal_descriptive_analysis import analyze_temporal_dataset
analyze_temporal_dataset(df, time_col="Date")

{'time_col': 'Date',
 'inferred_unit': 'hours',
 'shape': (8760, 14),
 'time_span_hours': 16752.0,
 'sampling_freq_hours': count      0.039722
 mean     117.146853
 std      219.085743
 min       24.000000
 25%       24.000000
 50%       24.000000
 75%       24.000000
 max      744.000000
 Name: Date, dtype: float64}

In [ ]:
# Variation is hourly for all non-seasonal features
df.groupby(df.Date.astype(str)).nunique().max().sort_values()

Date                          1
Seasons                       1
Holiday                       1
Functioning Day               2
Rainfall(mm)                 15
Snowfall (cm)                16
Solar Radiation (MJ/m2)      16
Wind speed (m/s)             22
Humidity(%)                  23
Dew point temperature(°C)    24
Hour                         24
Temperature(°C)              24
Rented Bike Count            24
Visibility (10m)             24
dtype: int64

In [3]:
df

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,30/11/2018,1003,19,4.2,34,2.6,1894,-10.3,0.0,0.0,0.0,Autumn,No Holiday,Yes
8756,30/11/2018,764,20,3.4,37,2.3,2000,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8757,30/11/2018,694,21,2.6,39,0.3,1968,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8758,30/11/2018,712,22,2.1,41,1.0,1859,-9.8,0.0,0.0,0.0,Autumn,No Holiday,Yes


In [ ]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For IID Data
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

# -- For Grouped Non-IID data
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

# -- For Temporal non-IID data -> manual processing required
# splits = {
#     repeat_i: {
#         fold_i: (train_idx, test_idx),
#     }
# }

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)